# HumorVibes S/R/E/B ablation court

A fail-closed judge-evidence run: fixed component ablations against real Humicroedit human grades, paired original/shuffled controls, explicit failure cases, and a runtime/provenance receipt. Gemma-2-2B supplies teacher-forced log-probabilities and the declared bad-surprise persona judgment.

In [ ]:
import os
from pathlib import Path
source_dir = Path('/kaggle/working/humorvibes_source')
source_dir.mkdir(parents=True, exist_ok=True)
(source_dir / 'mesh_signals.py').write_text('"""Measured humor signals from a causal LM (see THEORY.md).\n\nThe theory says a joke is a controlled prediction error with a cheap, permitted\nrepair. This module measures those quantities instead of asking a model to rate\nthem:\n\n- S  surprise    = token surprisal (nats) of the punchline given the setup\n- R  resolution  = surprisal collapse when the hidden frame is made explicit\n- E  efficiency  = resolution per token of frame hint (the ATP constraint)\n- B  bad surprise = persona-conditioned meta-mesh collision (canonical definition)\n\nProviders:\n- TransformersProvider: true logprobs from a local/Kaggle Gemma checkpoint.\n  Only selected when explicitly requested or when running inside Kaggle,\n  because loading a 2B model is a deliberate act, not a default.\n- OllamaProvider: generation + JSON judging via a Gemma served by Ollama;\n  uses logprobs when the server exposes them.\n- OfflineStub: deterministic heuristics so the UI/CLI stay demoable with no\n  model. Every result it returns is flagged measured=False.\n"""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport os\nimport re\nimport urllib.error\nimport urllib.request\nfrom dataclasses import dataclass, field, asdict\nfrom pathlib import Path\nfrom typing import Any, Protocol\n\nfrom humor_mesh import CANONICAL_BAD_SURPRISE_DEFINITION, extract_json_object\n\n# Surprise sweet band, in nats of mean punchline surprisal. Below the band the\n# continuation was predictable; above it no frame is likely to absorb the error.\nS_BAND_LOW = 1.2\nS_BAND_HIGH = 5.5\nS_BAND_PEAK = 3.0\n\nWEIGHTS = {"surprise": 0.30, "resolution": 0.35, "efficiency": 0.15, "benign": 0.20}\n\n\n@dataclass\nclass SurprisalProfile:\n    tokens: list[str]\n    nlls: list[float]\n    measured: bool\n\n    @property\n    def mean(self) -> float:\n        return sum(self.nlls) / len(self.nlls) if self.nlls else 0.0\n\n    @property\n    def peak(self) -> float:\n        return max(self.nlls) if self.nlls else 0.0\n\n\n@dataclass\nclass PersonaReport:\n    persona: str\n    collision: float  # 0-10 judged meta-mesh collision under the canonical definition\n    colliding_model: str\n    note: str\n    surprise_shift: float  # persona-conditioned S minus baseline S\n    measured: bool\n\n\n@dataclass\nclass HumorSignals:\n    setup: str\n    punchline: str\n    frame_hint: str\n    surprise_mean: float\n    surprise_peak: float\n    resolution: float          # NET resolution: frame collapse minus decoy-hint collapse\n    efficiency: float\n    resolution_raw: float = 0.0\n    resolution_null: float = 0.0  # decoy-hint collapse — conditioning on ANY text lowers NLL a bit\n    personas: list[PersonaReport] = field(default_factory=list)\n    measured: bool = True\n    profile: SurprisalProfile | None = None\n\n    @property\n    def bad_surprise(self) -> float:\n        if not self.personas:\n            return 0.0\n        return max(p.collision for p in self.personas)\n\n    @property\n    def surprise_score(self) -> float:\n        """Inverted U on RESIDUAL surprise. Corpus-lab calibration (2026-07-04):\n        puns measure S=7-9 with R up to 3.4 — far above the raw band — because a\n        strong frame absorbs the error. Leftover error is confusion; resolved\n        error is a pun. So above the band we judge S − R, not S."""\n        s = self.surprise_mean\n        if s >= S_BAND_HIGH:\n            s = max(S_BAND_PEAK, s - self.resolution)  # let the frame absorb the excess\n        if s <= S_BAND_LOW or s >= S_BAND_HIGH:\n            return 0.0\n        if s <= S_BAND_PEAK:\n            return (s - S_BAND_LOW) / (S_BAND_PEAK - S_BAND_LOW)\n        return (S_BAND_HIGH - s) / (S_BAND_HIGH - S_BAND_PEAK)\n\n    @property\n    def resolution_score(self) -> float:\n        return 1.0 - math.exp(-max(0.0, self.resolution) / 1.5)\n\n    @property\n    def efficiency_score(self) -> float:\n        return 1.0 - math.exp(-max(0.0, self.efficiency) / 0.15)\n\n    @property\n    def benign_score(self) -> float:\n        return 1.0 - min(10.0, self.bad_surprise) / 10.0\n\n    @property\n    def laugh_score(self) -> float:\n        return round(\n            100.0\n            * (\n                WEIGHTS["surprise"] * self.surprise_score\n                + WEIGHTS["resolution"] * self.resolution_score\n                + WEIGHTS["efficiency"] * self.efficiency_score\n                + WEIGHTS["benign"] * self.benign_score\n            ),\n            1,\n        )\n\n    @property\n    def failure_mode(self) -> str:\n        """Name which of the theory\'s four conditions failed hardest."""\n        if self.bad_surprise >= 6.0:\n            worst = max(self.personas, key=lambda p: p.collision)\n            return (\n                "bad-surprise: the frame collides with a high-authority internal model "\n                f"for \'{worst.persona}\' ({worst.colliding_model or \'unspecified\'})"\n            )\n        if self.surprise_mean <= S_BAND_LOW:\n            return "predictable: the punchline is what the supervisor already expected"\n        if self.surprise_mean >= S_BAND_HIGH and self.resolution < 0.5:\n            return "nonsense: high prediction error with no reachable frame"\n        if self.resolution < 0.5:\n            return "no re-route: the frame does not actually explain the punchline"\n        if self.efficiency < 0.03:\n            return "too expensive: the frame exists but costs too much to reach (dissected frog)"\n        return "laugh region: surprising, resolvable, affordable, permitted"\n\n    def to_dict(self) -> dict[str, Any]:\n        data = asdict(self)\n        data.pop("profile", None)\n        data.update(\n            surprise_score=round(self.surprise_score, 3),\n            resolution_score=round(self.resolution_score, 3),\n            efficiency_score=round(self.efficiency_score, 3),\n            benign_score=round(self.benign_score, 3),\n            bad_surprise=round(self.bad_surprise, 2),\n            laugh_score=self.laugh_score,\n            failure_mode=self.failure_mode,\n        )\n        return data\n\n\nclass SignalProvider(Protocol):\n    name: str\n\n    def nll_tokens(self, context: str, continuation: str) -> SurprisalProfile: ...\n\n    def generate(self, prompt: str, *, temperature: float = 0.8, max_tokens: int = 220) -> str: ...\n\n    def judge_json(self, prompt: str) -> dict[str, Any] | None: ...\n\n\n# --------------------------------------------------------------------------\n# Providers\n# --------------------------------------------------------------------------\nclass TransformersProvider:\n    """True logprobs from a Gemma checkpoint via transformers."""\n\n    name = "transformers"\n\n    def __init__(self, model_path: str | None = None) -> None:\n        import torch  # noqa: F401  (lazy heavy imports on purpose)\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n\n        self.torch = torch\n        path = model_path or os.environ.get("GEMMA_MODEL_PATH") or self._find_kaggle_gemma() or "google/gemma-2-2b-it"\n        self.tokenizer = AutoTokenizer.from_pretrained(path)\n        self.model = None\n        if torch.cuda.is_available():\n            try:\n                model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16, device_map="auto")\n                model.eval()\n                with torch.no_grad():  # probe: some assignments lack kernels for this arch\n                    model(torch.tensor([[self.tokenizer.bos_token_id or 2]]).to(model.device))\n                self.model = model\n            except Exception:\n                self.model = None\n                torch.cuda.empty_cache()\n        if self.model is None:\n            self.model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float32)\n            self.model.eval()\n\n    @staticmethod\n    def _find_kaggle_gemma() -> str | None:\n        root = Path("/kaggle/input")\n        if not root.exists():\n            return None\n        hits = sorted(root.glob("**/config.json"))\n        for hit in hits:\n            if "gemma" in str(hit).lower():\n                return str(hit.parent)\n        return None\n\n    def nll_tokens(self, context: str, continuation: str) -> SurprisalProfile:\n        torch = self.torch\n        ctx_ids = self.tokenizer(context, return_tensors="pt").input_ids\n        cont_ids = self.tokenizer(continuation, add_special_tokens=False, return_tensors="pt").input_ids\n        full = torch.cat([ctx_ids, cont_ids], dim=1).to(self.model.device)\n        with torch.no_grad():\n            logits = self.model(full).logits\n        logprobs = torch.log_softmax(logits.float(), dim=-1)\n        n_ctx = ctx_ids.shape[1]\n        nlls: list[float] = []\n        toks: list[str] = []\n        for i in range(cont_ids.shape[1]):\n            pos = n_ctx + i\n            tok_id = int(full[0, pos])\n            nlls.append(float(-logprobs[0, pos - 1, tok_id]))\n            toks.append(self.tokenizer.decode([tok_id]))\n        return SurprisalProfile(tokens=toks, nlls=nlls, measured=True)\n\n    def generate(self, prompt: str, *, temperature: float = 0.8, max_tokens: int = 220) -> str:\n        torch = self.torch\n        chat = [{"role": "user", "content": prompt}]\n        ids = self.tokenizer.apply_chat_template(chat, return_tensors="pt", add_generation_prompt=True)\n        if not torch.is_tensor(ids):  # newer transformers return a BatchEncoding\n            ids = ids["input_ids"]\n        ids = ids.to(self.model.device)\n        with torch.no_grad():\n            out = self.model.generate(\n                ids,\n                max_new_tokens=max_tokens,\n                do_sample=temperature > 0,\n                temperature=max(temperature, 1e-3),\n                top_p=0.95,\n                pad_token_id=self.tokenizer.eos_token_id,\n            )\n        return self.tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()\n\n    def judge_json(self, prompt: str) -> dict[str, Any] | None:\n        return extract_json_object(self.generate(prompt, temperature=0.2, max_tokens=300))\n\n\nclass OllamaProvider:\n    """Gemma via a local/remote Ollama server; logprobs when supported."""\n\n    name = "ollama"\n\n    def __init__(self) -> None:\n        self.model = os.environ.get("GEMMA_MODEL", "gemma3:4b")\n        self.host = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434").rstrip("/")\n        self.think = os.environ.get("GEMMA_THINK", "0").strip().lower() in {\n            "1", "true", "yes", "on"\n        }\n\n    def _post(self, payload: dict[str, Any]) -> dict[str, Any] | None:\n        req = urllib.request.Request(\n            f"{self.host}/api/generate",\n            data=json.dumps(payload).encode("utf-8"),\n            headers={"Content-Type": "application/json"},\n            method="POST",\n        )\n        try:\n            with urllib.request.urlopen(req, timeout=180) as resp:\n                return json.loads(resp.read().decode("utf-8"))\n        except (urllib.error.URLError, TimeoutError, json.JSONDecodeError):\n            return None\n\n    def nll_tokens(self, context: str, continuation: str) -> SurprisalProfile:\n        # Ollama does not expose teacher-forced continuation logprobs; fall back\n        # to the stub\'s heuristic profile but keep generation/judging real.\n        return OfflineStub().nll_tokens(context, continuation)\n\n    def generate(self, prompt: str, *, temperature: float = 0.8, max_tokens: int = 220) -> str:\n        data = self._post(\n            {\n                "model": self.model,\n                "prompt": prompt,\n                "stream": False,\n                # Gemma 4 can spend ``num_predict`` on hidden reasoning and\n                # truncate the requested visible candidates. Humor generation\n                # is a constrained writing task, so thinking is opt-in.\n                "think": self.think,\n                "options": {"temperature": temperature, "num_predict": max_tokens},\n            }\n        )\n        return str((data or {}).get("response", "")).strip()\n\n    def judge_json(self, prompt: str) -> dict[str, Any] | None:\n        return extract_json_object(self.generate(prompt, temperature=0.2, max_tokens=320))\n\n\nclass OpenAICompatProvider:\n    """Hosted Gemma (or other) via any OpenAI-compatible endpoint — NVIDIA NIM\n    (free, hosts google/gemma-2-9b-it), Ollama Cloud (gemma3), Mistral, etc.\n    Generation + judging only; teacher-forced logprobs are not exposed by these\n    APIs, so measurement falls back to the stub and is flagged unmeasured."""\n\n    name = "openai-compat"\n\n    def __init__(self) -> None:\n        self.base = os.environ.get("GEMMA_OPENAI_BASE_URL", "https://integrate.api.nvidia.com/v1").rstrip("/")\n        self.model = os.environ.get("GEMMA_OPENAI_MODEL", "google/gemma-2-9b-it")\n        self.key = ""\n        for env in (os.environ.get("GEMMA_OPENAI_KEY_ENV", ""), "NVIDIA_API_KEY",\n                    "OLLAMA_CLOUD_API_KEY", "ADVISOR_LLM_API_KEY", "MISTRAL_API_KEY", "OPENAI_API_KEY"):\n            if env and os.environ.get(env):\n                self.key = os.environ[env]\n                break\n\n    def nll_tokens(self, context: str, continuation: str) -> SurprisalProfile:\n        return OfflineStub().nll_tokens(context, continuation)\n\n    def generate(self, prompt: str, *, temperature: float = 0.8, max_tokens: int = 220) -> str:\n        req = urllib.request.Request(\n            f"{self.base}/chat/completions",\n            data=json.dumps({"model": self.model, "temperature": temperature, "max_tokens": max_tokens,\n                             "messages": [{"role": "user", "content": prompt}]}).encode("utf-8"),\n            headers={"Content-Type": "application/json", "Authorization": f"Bearer {self.key}"},\n            method="POST",\n        )\n        try:\n            with urllib.request.urlopen(req, timeout=120) as resp:\n                data = json.loads(resp.read().decode("utf-8"))\n            return str(data["choices"][0]["message"]["content"]).strip()\n        except Exception:\n            return ""\n\n    def judge_json(self, prompt: str) -> dict[str, Any] | None:\n        return extract_json_object(self.generate(prompt, temperature=0.2, max_tokens=350))\n\n\nclass PollinationsProvider:\n    """Keyless hosted text generation (verified 2026-07-04: GET\n    https://text.pollinations.ai/<prompt> answers with no auth). Rate-limited\n    community service — treat as a bonus lane for judging/writing, never the\n    core; measurement still falls back to the stub (no logprobs)."""\n\n    name = "pollinations"\n\n    def nll_tokens(self, context: str, continuation: str) -> SurprisalProfile:\n        return OfflineStub().nll_tokens(context, continuation)\n\n    def generate(self, prompt: str, *, temperature: float = 0.8, max_tokens: int = 220) -> str:\n        import urllib.parse\n\n        url = "https://text.pollinations.ai/" + urllib.parse.quote(prompt[:1800])\n        req = urllib.request.Request(url, headers={"User-Agent": "HumorVibes research"})\n        try:\n            with urllib.request.urlopen(req, timeout=60) as resp:\n                return resp.read().decode("utf-8", "replace").strip()[: max_tokens * 6]\n        except (urllib.error.URLError, TimeoutError):\n            return ""\n\n    def judge_json(self, prompt: str) -> dict[str, Any] | None:\n        return extract_json_object(self.generate(prompt, temperature=0.2, max_tokens=350))\n\n\nclass OfflineStub:\n    """Deterministic pseudo-signals; keeps demos alive with measured=False."""\n\n    name = "offline"\n\n    def nll_tokens(self, context: str, continuation: str) -> SurprisalProfile:\n        ctx_words = {w.lower().strip(".,!?") for w in context.split()}\n        toks, nlls = [], []\n        for word in continuation.split():\n            base = 1.0 + min(4.0, len(word) * 0.28)\n            if word.lower().strip(".,!?") in ctx_words:\n                base *= 0.35  # repeated words are predictable\n            digest = sum(ord(c) for c in word) % 7\n            toks.append(word)\n            nlls.append(round(base + digest * 0.22, 3))\n        return SurprisalProfile(tokens=toks, nlls=nlls, measured=False)\n\n    def generate(self, prompt: str, *, temperature: float = 0.8, max_tokens: int = 220) -> str:\n        return ""\n\n    def judge_json(self, prompt: str) -> dict[str, Any] | None:\n        return None\n\n\ndef get_provider(kind: str | None = None) -> SignalProvider:\n    kind = (kind or os.environ.get("GEMMA_PROVIDER", "")).strip().lower()\n    if kind == "transformers" or (not kind and Path("/kaggle/input").exists()):\n        try:\n            return TransformersProvider()\n        except Exception:\n            return OfflineStub()\n    if kind == "ollama":\n        return OllamaProvider()\n    if kind in ("openai", "openai-compat", "nvidia", "ollama-cloud", "mistral"):\n        return OpenAICompatProvider()\n    if kind == "pollinations":\n        return PollinationsProvider()\n    return OfflineStub()\n\n\n# --------------------------------------------------------------------------\n# Signal computation\n# --------------------------------------------------------------------------\nFRAME_PROMPT = (\n    "A joke works because a hidden frame reinterprets the punchline. The frame is the fact that,"\n    " once stated, makes the punchline the OBVIOUS next thing to say.\\n\\n"\n    "Example 1 — Setup: I told my therapist about my fear of speed bumps.\\n"\n    "Punchline: She said I\'m slowly getting over it.\\n"\n    "Frame: \'Getting over it\' is literal — the car physically drives over the speed bumps slowly.\\n\\n"\n    "Example 2 — Setup: My grandfather has the heart of a lion\\n"\n    "Punchline: and a lifetime ban from the zoo.\\n"\n    "Frame: He literally stole a lion\'s heart from the zoo, not the metaphor for bravery.\\n\\n"\n    "Now — Setup: {setup}\\n"\n    "Punchline: {punchline}\\n"\n    "Frame (ONE short sentence, no preamble, no quotes; if there is no such fact, write NONE):"\n)\n\nPERSONA_JUDGE_PROMPT = (\n    "You evaluate one specific failure mode of humor, defined canonically as:\\n"\n    f"\\"{CANONICAL_BAD_SURPRISE_DEFINITION}\\"\\n\\n"\n    "Audience persona: {persona}\\n"\n    "Joke setup: {setup}\\n"\n    "Punchline: {punchline}\\n"\n    "Reframe the punchline relies on: {frame}\\n\\n"\n    "Does that reframe collide with an internal model this audience uses with "\n    "override authority (identity, moral core, worldview)? Mild discomfort or "\n    "edginess is NOT a collision. Answer as JSON only: "\n    \'{{"collision": 0-10, "colliding_model": "name the internal model or empty", \'\n    \'"note": "one sentence"}}\'\n)\n\n\ndef split_setup_punchline(text: str) -> tuple[str, str]:\n    """Best-effort split of a one-block joke into setup/punchline."""\n    text = text.strip()\n    for sep in ["\\n", ". ", "? ", "! ", " — ", " - ", ": ", "; "]:\n        if sep in text:\n            head, tail = text.rsplit(sep, 1)\n            if len(tail.split()) >= 2:\n                return head + sep.strip(), tail.strip()\n    words = text.split()\n    cut = max(1, int(len(words) * 0.7))\n    return " ".join(words[:cut]), " ".join(words[cut:])\n\n\ndef compute_signals(\n    provider: SignalProvider,\n    setup: str,\n    punchline: str,\n    frame_hint: str | None = None,\n    personas: list[str] | None = None,\n) -> HumorSignals:\n    setup = setup.strip()\n    punchline = punchline.strip()\n    base = provider.nll_tokens(setup + "\\n", " " + punchline)\n\n    hint = (frame_hint or "").strip()\n    if not hint:\n        hint = provider.generate(FRAME_PROMPT.format(setup=setup, punchline=punchline), temperature=0.3, max_tokens=60)\n        hint = hint.splitlines()[0].strip() if hint else ""\n    if hint:\n        framed = provider.nll_tokens(setup + "\\n(" + hint + ")\\n", " " + punchline)\n        resolution_raw = max(0.0, base.mean - framed.mean)\n        # Leaky-frame guard (zoo-lab finding 2026-07-04): a "frame" that lexically\n        # contains the punchline predicts its text without reframing anything —\n        # confabulations can beat the generic decoy this way. Discount by content-\n        # word overlap between hint and the punchline\'s NOVEL words only — words\n        # the punchline shares with the setup are fair reuse, not a leak (refined\n        # 2026-07-03: penalizing setup-word reuse was punishing legitimate frames).\n        punch_words = {w.lower().strip(".,!?\\"\'") for w in punchline.split() if len(w) > 3}\n        setup_words = {w.lower().strip(".,!?\\"\'") for w in setup.split() if len(w) > 3}\n        novel_punch_words = punch_words - setup_words\n        hint_words = {w.lower().strip(".,!?\\"\'") for w in hint.split() if len(w) > 3}\n        leak = len(novel_punch_words & hint_words) / max(1, len(novel_punch_words))\n        if leak > 0.4:\n            resolution_raw *= max(0.0, 1.0 - leak)\n        # Null control (house doctrine: every localized effect needs one).\n        # Conditioning on ANY specific text lowers NLL slightly — and a model\n        # asked to "find the frame" of nonsense will confabulate one. Subtract\n        # the collapse produced by an equal-length decoy hint.\n        decoy = "It turns out this is really about quarterly regional cheese sales figures."\n        nulled = provider.nll_tokens(setup + "\\n(" + decoy + ")\\n", " " + punchline)\n        resolution_null = max(0.0, base.mean - nulled.mean)\n        resolution = max(0.0, resolution_raw - resolution_null)\n        hint_tokens = max(1, len(hint.split()))\n        efficiency = resolution / hint_tokens\n    else:\n        resolution_raw, resolution_null, resolution, efficiency = 0.0, 0.0, 0.0, 0.0\n\n    reports: list[PersonaReport] = []\n    for persona in personas or []:\n        judged = provider.judge_json(\n            PERSONA_JUDGE_PROMPT.format(persona=persona, setup=setup, punchline=punchline, frame=hint or "unknown")\n        )\n        persona_ctx = f"Audience: {persona}.\\n{setup}\\n"\n        shifted = provider.nll_tokens(persona_ctx, " " + punchline)\n        if judged is None:\n            reports.append(\n                PersonaReport(\n                    persona=persona,\n                    collision=0.0,\n                    colliding_model="",\n                    note="no judge available (offline)",\n                    surprise_shift=round(shifted.mean - base.mean, 3),\n                    measured=False,\n                )\n            )\n        else:\n            reports.append(\n                PersonaReport(\n                    persona=persona,\n                    collision=float(max(0.0, min(10.0, float(judged.get("collision", 0))))),\n                    colliding_model=str(judged.get("colliding_model", "")).strip(),\n                    note=str(judged.get("note", "")).strip(),\n                    surprise_shift=round(shifted.mean - base.mean, 3),\n                    measured=base.measured,\n                )\n            )\n\n    return HumorSignals(\n        setup=setup,\n        punchline=punchline,\n        frame_hint=hint,\n        surprise_mean=round(base.mean, 3),\n        surprise_peak=round(base.peak, 3),\n        resolution=round(resolution, 3),\n        efficiency=round(efficiency, 4),\n        resolution_raw=round(resolution_raw, 3),\n        resolution_null=round(resolution_null, 3),\n        personas=reports,\n        measured=base.measured,\n        profile=base,\n    )\n\n\ndef sparkline(nlls: list[float]) -> str:\n    """Tiny text sparkline of per-token surprisal for CLI/notebook output."""\n    if not nlls:\n        return ""\n    blocks = "▁▂▃▄▅▆▇█"\n    lo, hi = min(nlls), max(nlls)\n    span = (hi - lo) or 1.0\n    return "".join(blocks[int((v - lo) / span * (len(blocks) - 1))] for v in nlls)\n', encoding='utf-8')
(source_dir / 'humor_mesh.py').write_text('"""Humor mesh scoring primitives for HumorVibes.\n\nThe app can run in two modes:\n- Gemma mode: Gemma returns structured JSON through the prompts in prompts/.\n- Fallback mode: deterministic local heuristics keep the prototype demoable.\n"""\nfrom __future__ import annotations\n\nimport json\nimport re\nfrom dataclasses import asdict, dataclass\nfrom typing import Any\n\n\nCANONICAL_BAD_SURPRISE_DEFINITION = (\n    "Bad surprise is poorly defined, a bad surprise is a surprise that contradicts "\n    "with internal models within a human brain that are so strong they override "\n    "logic and are some of the primary drivers of a person\'s perception, "\n    "understanding, and good/bad/moral/ethical views of the world. So basically, "\n    "a surprise is not good if it disagrees with something that is already "\n    "overriding logic or a surprise is not good it if disagrees with a nearly "\n    "overwhelming generalization engine in a human mind that has significant "\n    "overriding power to override logic, promote other false generalizations, "\n    "and is the primary feature used to reduce surprise in that person\'s mind."\n)\n\n\nMESH_DIMENSIONS = [\n    "comedic_structure",\n    "audience_reaction_fit",\n    "timing",\n    "surprise",\n    "cultural_context",\n    "preference_fit",\n    "truth_alignment",\n    "bad_surprise_risk",\n]\n\n\n@dataclass\nclass MeshScore:\n    candidate: str\n    comedic_structure: int\n    audience_reaction_fit: int\n    timing: int\n    surprise: int\n    cultural_context: int\n    preference_fit: int\n    truth_alignment: int\n    bad_surprise_risk: int\n    risk_flags: list[str]\n    why_it_works: str\n    repair_strategy: str\n    repaired_candidate: str\n\n    @property\n    def total(self) -> float:\n        positive = (\n            self.comedic_structure\n            + self.audience_reaction_fit\n            + self.timing\n            + self.surprise\n            + self.cultural_context\n            + self.preference_fit\n            + self.truth_alignment\n        )\n        return round((positive / 7.0) - (0.75 * self.bad_surprise_risk), 2)\n\n\ndef clamp_score(value: Any) -> int:\n    try:\n        return max(0, min(10, int(round(float(value)))))\n    except Exception:\n        return 0\n\n\ndef normalize_mesh_record(record: dict[str, Any], fallback_candidate: str = "") -> MeshScore:\n    return MeshScore(\n        candidate=str(record.get("candidate") or fallback_candidate).strip(),\n        comedic_structure=clamp_score(record.get("comedic_structure", record.get("setup_clarity", 5))),\n        audience_reaction_fit=clamp_score(record.get("audience_reaction_fit", 5)),\n        timing=clamp_score(record.get("timing", 5)),\n        surprise=clamp_score(record.get("surprise", 5)),\n        cultural_context=clamp_score(record.get("cultural_context", record.get("cultural_fit", 5))),\n        preference_fit=clamp_score(record.get("preference_fit", 5)),\n        truth_alignment=clamp_score(record.get("truth_alignment", 7)),\n        bad_surprise_risk=clamp_score(record.get("bad_surprise_risk", 3)),\n        risk_flags=[str(x) for x in record.get("risk_flags", []) if str(x).strip()],\n        why_it_works=str(record.get("why_it_works", "")).strip(),\n        repair_strategy=str(record.get("repair_strategy", record.get("rewrite_suggestion", ""))).strip(),\n        repaired_candidate=str(record.get("repaired_candidate", record.get("rewrite", ""))).strip(),\n    )\n\n\ndef extract_json_object(text: str) -> dict[str, Any] | None:\n    text = text.strip()\n    if not text:\n        return None\n    try:\n        parsed = json.loads(text)\n        if isinstance(parsed, dict):\n            return parsed\n    except json.JSONDecodeError:\n        pass\n    match = re.search(r"\\{.*\\}", text, flags=re.DOTALL)\n    if not match:\n        return None\n    try:\n        parsed = json.loads(match.group(0))\n    except json.JSONDecodeError:\n        return None\n    return parsed if isinstance(parsed, dict) else None\n\n\ndef extract_candidates(text: str, limit: int = 5) -> list[str]:\n    """Recover distinct candidates from JSON, numbered, or bulleted output.\n\n    Local thinking-capable models occasionally wrap JSON in prose or switch\n    from the requested numbering style to bullets.  Parsing that variation is\n    a transport concern; it must not silently turn a successful generation\n    into the deterministic fallback path.\n    """\n    raw = str(text or "").strip()\n    if not raw:\n        return []\n\n    values: Any = None\n    try:\n        parsed = json.loads(raw)\n        if isinstance(parsed, list):\n            values = parsed\n        elif isinstance(parsed, dict):\n            values = parsed.get("jokes", parsed.get("candidates"))\n    except json.JSONDecodeError:\n        parsed = extract_json_object(raw)\n        if parsed:\n            values = parsed.get("jokes", parsed.get("candidates"))\n\n    candidates: list[str] = []\n    if isinstance(values, list):\n        candidates.extend(str(value).strip() for value in values)\n    elif isinstance(values, str):\n        candidates.append(values.strip())\n\n    if not candidates:\n        marker = re.compile(r"(?m)^\\s*(?:\\d{1,2}[.)]|[-*])\\s+")\n        matches = list(marker.finditer(raw))\n        for index, match in enumerate(matches):\n            end = matches[index + 1].start() if index + 1 < len(matches) else len(raw)\n            candidates.append(raw[match.end():end].strip())\n\n    if not candidates:\n        candidates.extend(\n            line.strip().strip("`")\n            for line in raw.splitlines()\n            if line.strip() and line.strip() not in {"```", "```json"}\n        )\n\n    unique: list[str] = []\n    seen: set[str] = set()\n    for candidate in candidates:\n        cleaned = re.sub(r"\\s+", " ", candidate).strip().strip(\'"\')\n        key = cleaned.casefold()\n        if cleaned and key not in seen:\n            seen.add(key)\n            unique.append(cleaned)\n        if len(unique) >= max(1, limit):\n            break\n    return unique\n\n\ndef fallback_generate(prompt: str, audience: str, count: int = 3) -> list[str]:\n    topic = prompt.strip().rstrip(".") or "everyday life"\n    topic = re.sub(r"^(please\\s+)?(make|write|generate)\\s+(a\\s+)?joke\\s+(about|on)\\s+", "", topic, flags=re.I)\n    topic = re.sub(r"\\bkeep it\\b.*$", "", topic, flags=re.I).strip(" .") or "everyday life"\n    audience = audience.strip() or "a general audience"\n    return [\n        f"For {audience}, {topic} is like a meeting invite: the setup is short, but somehow the follow-up has 47 stakeholders.",\n        f"I tried to make {topic} more efficient, but it formed a committee to evaluate whether efficiency aligned with its roadmap.",\n        f"The problem with {topic} is not that it surprises people; it surprises the calendar, then asks the calendar to circle back.",\n    ][: max(1, min(count, 5))]\n\n\ndef fallback_evaluate(candidate: str, prompt: str, audience: str, preferences: str = "") -> MeshScore:\n    words = candidate.split()\n    lower = candidate.lower()\n    has_turn = any(token in lower for token in ["but", "then", "somehow", "instead", "only"])\n    has_specifics = len(set(re.findall(r"[a-zA-Z]{4,}", candidate))) >= 8\n    too_long = len(words) > 35\n    too_short = len(words) < 8\n    truth_risk = any(token in lower for token in ["always", "never", "everyone", "proves"])\n    cruelty_risk = any(token in lower for token in ["stupid", "idiot", "hate", "loser"])\n\n    risk_flags = []\n    if too_long:\n        risk_flags.append("timing: setup may be too long")\n    if too_short:\n        risk_flags.append("structure: premise may be underdeveloped")\n    if truth_risk:\n        risk_flags.append("truth alignment: overgeneralized claim")\n    if cruelty_risk:\n        risk_flags.append("targeting: avoid lazy personal attack")\n\n    bad_surprise_risk = 2 + (3 if truth_risk else 0) + (3 if cruelty_risk else 0)\n    if preferences and any(term in lower for term in preferences.lower().split()):\n        preference_fit = 8\n    else:\n        preference_fit = 6\n\n    repaired = candidate\n    if too_long:\n        repaired = re.sub(r",?\\s+and\\s+", ", then ", candidate, count=1)\n    if cruelty_risk:\n        repaired = repaired.replace("stupid", "overconfident").replace("idiot", "calendar invite")\n    if truth_risk:\n        repaired = repaired.replace("always", "sometimes").replace("never", "rarely")\n\n    return MeshScore(\n        candidate=candidate,\n        comedic_structure=8 if has_turn and has_specifics else 5,\n        audience_reaction_fit=7 if audience else 5,\n        timing=4 if too_long or too_short else 8,\n        surprise=8 if has_turn else 5,\n        cultural_context=7 if audience else 5,\n        preference_fit=preference_fit,\n        truth_alignment=5 if truth_risk else 8,\n        bad_surprise_risk=clamp_score(bad_surprise_risk),\n        risk_flags=risk_flags,\n        why_it_works=(\n            "The candidate has a recognizable setup and a turn."\n            if has_turn\n            else "The candidate has a premise but needs a clearer expectation shift."\n        ),\n        repair_strategy=(\n            "Preserve the turn while reducing overgeneralization, personal attack, or excess setup."\n            if risk_flags\n            else "Keep the structure; adapt wording to the audience."\n        ),\n        repaired_candidate=repaired,\n    )\n\n\ndef best_candidate(scores: list[MeshScore]) -> MeshScore | None:\n    return max(scores, key=lambda s: s.total, default=None)\n\n\ndef to_json(score: MeshScore) -> str:\n    data = asdict(score)\n    data["total"] = score.total\n    return json.dumps(data, indent=2)\n', encoding='utf-8')
os.environ['HUMORVIBES_SOURCE_DIR'] = str(source_dir)
print('vendored exact HumorVibes signal source:', source_dir)


In [ ]:
#!/usr/bin/env python3
"""HumorVibes judge-evidence court for Kaggle.

Runs a fixed-weight S/R/E/B leave-one-component ablation against real Humicroedit
human grades, plus paired original-headline and shuffled-edit controls.  The
script writes per-item evidence, failure cases, a figure, and a runtime receipt.

This is designed for the private Kaggle kernel `humorvibes-ablation-court` with
the exact signal source vendored into the notebook and google/gemma-2-2b-it attached.
"""

from __future__ import annotations

import csv
import glob
import hashlib
import io
import json
import os
import platform
import random
import re
import sys
import time
import urllib.request
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr, wilcoxon


SEED = 20260712
N_HUMAN = 120
N_CONTROL = 40
BOOTSTRAPS = 1000
DATA_URL = "https://cs.rochester.edu/u/nhossain/humicroedit/semeval-2020-task-7-data.zip"
MODEL_SOURCE = "google/gemma-2/transformers/gemma-2-2b-it/2"
KERNEL_ID = "taylorsamarel/humorvibes-ablation-court"
PERSONA = "a broad U.S. news audience including people or groups named in the headline"
WEIGHTS = {"S": 0.30, "R": 0.35, "E": 0.15, "B": 0.20}


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: str | Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as fh:
        for chunk in iter(lambda: fh.read(4 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def json_dump(path: str | Path, payload: Any) -> None:
    Path(path).write_text(json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=False) + "\n", encoding="utf-8")


def original_word(text: str) -> str:
    match = re.search(r"<([^/>]+)/>", str(text))
    return match.group(1) if match else ""


def apply_edit(text: str, replacement: str) -> str:
    return re.sub(r"<[^/>]+/>", str(replacement), str(text))


def fixed_score(frame: pd.DataFrame, components: list[str]) -> np.ndarray:
    denominator = sum(WEIGHTS[name] for name in components)
    if denominator <= 0:
        raise ValueError("At least one component is required")
    score = np.zeros(len(frame), dtype=np.float64)
    for name in components:
        score += WEIGHTS[name] * frame[f"{name}_score"].to_numpy(float)
    return 100.0 * score / denominator


def correlations(x: np.ndarray, y: np.ndarray) -> dict[str, float]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return {"pearson": 0.0, "pearson_p": 1.0, "spearman": 0.0, "spearman_p": 1.0}
    pearson = pearsonr(x, y)
    spearman = spearmanr(x, y)
    return {
        "pearson": float(pearson.statistic),
        "pearson_p": float(pearson.pvalue),
        "spearman": float(spearman.statistic),
        "spearman_p": float(spearman.pvalue),
    }


def bootstrap_spearman_ci(x: np.ndarray, y: np.ndarray, seed: int = SEED, rounds: int = BOOTSTRAPS) -> list[float]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    rng = np.random.default_rng(seed)
    values: list[float] = []
    for _ in range(rounds):
        idx = rng.integers(0, len(x), size=len(x))
        if np.std(x[idx]) == 0 or np.std(y[idx]) == 0:
            continue
        value = spearmanr(x[idx], y[idx]).statistic
        if np.isfinite(value):
            values.append(float(value))
    return [float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))] if values else [0.0, 0.0]


def component_ablation(human: pd.DataFrame) -> dict[str, Any]:
    all_components = ["S", "R", "E", "B"]
    variants: dict[str, list[str]] = {"full_SREB": all_components}
    for dropped in all_components:
        variants[f"without_{dropped}"] = [name for name in all_components if name != dropped]
    for component in all_components:
        variants[f"only_{component}"] = [component]

    results: dict[str, Any] = {}
    grade = human["grade"].to_numpy(float)
    for offset, (name, components) in enumerate(variants.items()):
        score = fixed_score(human, components)
        result = correlations(score, grade)
        result["spearman_bootstrap_95ci"] = bootstrap_spearman_ci(score, grade, SEED + offset)
        result["components"] = components
        result["score_mean"] = float(np.mean(score))
        result["score_std"] = float(np.std(score))
        results[name] = result

    full_rho = results["full_SREB"]["spearman"]
    for name, result in results.items():
        result["spearman_delta_vs_full"] = float(result["spearman"] - full_rho)
    return results


def paired_control_court(frame: pd.DataFrame) -> dict[str, Any]:
    controls = frame[frame["control_set"]].copy()
    variants = ["human_edit", "original_headline", "shuffled_edit"]
    metrics = ["S_score", "R_score", "E_score", "B_score", "full_score"]
    summary: dict[str, Any] = {"n_complete_sets": 0, "variant_means": {}, "paired_tests": {}}
    complete_ids = []
    for item_id, group in controls.groupby("id"):
        if set(group["variant"]) == set(variants):
            complete_ids.append(item_id)
    controls = controls[controls["id"].isin(complete_ids)]
    summary["n_complete_sets"] = len(complete_ids)
    for variant in variants:
        block = controls[controls["variant"] == variant]
        summary["variant_means"][variant] = {
            metric: float(block[metric].mean()) for metric in metrics
        }

    for other in ("original_headline", "shuffled_edit"):
        tests = {}
        for metric in metrics:
            pivot = controls.pivot(index="id", columns="variant", values=metric).dropna()
            human_values = pivot["human_edit"].to_numpy(float)
            other_values = pivot[other].to_numpy(float)
            try:
                test = wilcoxon(human_values, other_values, zero_method="zsplit", alternative="greater")
                statistic, pvalue = float(test.statistic), float(test.pvalue)
            except ValueError:
                statistic, pvalue = 0.0, 1.0
            difference = human_values - other_values
            tests[metric] = {
                "mean_human_minus_control": float(np.mean(difference)),
                "median_human_minus_control": float(np.median(difference)),
                "wilcoxon_greater_statistic": statistic,
                "wilcoxon_greater_p": pvalue,
                "wins": int(np.sum(difference > 0)),
                "ties": int(np.sum(difference == 0)),
                "losses": int(np.sum(difference < 0)),
            }
        summary["paired_tests"][f"human_vs_{other}"] = tests
    return summary


def find_source_file(filename: str) -> Path:
    vendored = os.environ.get("HUMORVIBES_SOURCE_DIR")
    if vendored:
        candidate = Path(vendored) / filename
        if candidate.exists():
            return candidate
    hits = [Path(p) for p in glob.glob(f"/kaggle/input/**/{filename}", recursive=True)]
    if not hits:
        raise FileNotFoundError(f"Attached source is missing {filename}")
    return hits[0]


def load_provider() -> tuple[Any, dict[str, Any]]:
    mesh_path = find_source_file("mesh_signals.py")
    sys.path.insert(0, str(mesh_path.parent))
    os.environ["GEMMA_PROVIDER"] = "transformers"
    configs = [Path(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if "gemma" in p.lower()]
    if not configs:
        raise FileNotFoundError("Attached Gemma config.json not found")
    model_config = configs[0]
    os.environ["GEMMA_MODEL_PATH"] = str(model_config.parent)
    from mesh_signals import TransformersProvider

    provider = TransformersProvider(str(model_config.parent))
    import torch

    config_json = json.loads(model_config.read_text(encoding="utf-8"))
    model_name = str(config_json.get("_name_or_path") or config_json.get("model_type") or "unknown")
    evidence = {
        "provider_class": type(provider).__name__,
        "provider_name": provider.name,
        "model_source": MODEL_SOURCE,
        "model_config_path": str(model_config),
        "model_config_sha256": sha256_file(model_config),
        "model_config_name": model_name,
        "parameter_count": int(sum(parameter.numel() for parameter in provider.model.parameters())),
        "device": str(next(provider.model.parameters()).device),
        "dtype": str(next(provider.model.parameters()).dtype),
        "torch_version": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "mesh_signals_path": str(mesh_path),
        "mesh_signals_sha256": sha256_file(mesh_path),
        "humor_mesh_sha256": sha256_file(mesh_path.parent / "humor_mesh.py"),
        "source_delivery": "self_contained_notebook_vendor",
        "true_teacher_forced_logprobs": True,
    }
    if provider.name != "transformers":
        raise RuntimeError("Exact Gemma logprob provider was not engaged")
    return provider, evidence


def load_humicroedit() -> tuple[pd.DataFrame, dict[str, Any]]:
    request = urllib.request.Request(DATA_URL, headers={"User-Agent": "HumorVibes ablation research"})
    raw = urllib.request.urlopen(request, timeout=120).read()
    archive_sha = sha256_bytes(raw)
    archive = zipfile.ZipFile(io.BytesIO(raw))
    member = "data/task-1/train.csv"
    csv_bytes = archive.read(member)
    data = pd.read_csv(io.BytesIO(csv_bytes)).dropna(subset=["meanGrade"])
    sampled = data.sample(N_HUMAN, random_state=SEED).reset_index(drop=True)
    sampled["id"] = sampled["id"].astype(str)
    sampled["original_word"] = sampled["original"].map(original_word)
    sampled["human_text"] = [apply_edit(original, edit) for original, edit in zip(sampled["original"], sampled["edit"])]
    sampled["original_text"] = [apply_edit(original, word) for original, word in zip(sampled["original"], sampled["original_word"])]
    control_positions = np.linspace(0, len(sampled) - 1, N_CONTROL, dtype=int)
    sampled["control_set"] = False
    sampled.loc[control_positions, "control_set"] = True
    control_edits = sampled.loc[control_positions, "edit"].astype(str).tolist()
    rotated = control_edits[7:] + control_edits[:7]
    shuffled_by_id = dict(zip(sampled.loc[control_positions, "id"], rotated))
    sampled["shuffled_text"] = [
        apply_edit(original, shuffled_by_id.get(item_id, edit))
        for original, item_id, edit in zip(sampled["original"], sampled["id"], sampled["edit"])
    ]
    evidence = {
        "url": DATA_URL,
        "archive_sha256": archive_sha,
        "csv_member": member,
        "csv_sha256": sha256_bytes(csv_bytes),
        "source_rows": int(len(data)),
        "human_sample_rows": int(len(sampled)),
        "control_set_rows": int(sampled["control_set"].sum()),
        "sample_id_sha256": sha256_bytes("\n".join(sampled["id"]).encode("utf-8")),
        "sampling_seed": SEED,
    }
    return sampled, evidence


def measurement_jobs(sampled: pd.DataFrame) -> list[dict[str, Any]]:
    jobs: list[dict[str, Any]] = []
    for _, row in sampled.iterrows():
        base = {
            "id": str(row["id"]),
            "grade": float(row["meanGrade"]),
            "original": str(row["original"]),
            "edit": str(row["edit"]),
            "control_set": bool(row["control_set"]),
        }
        jobs.append({**base, "variant": "human_edit", "text": str(row["human_text"])})
        if row["control_set"]:
            jobs.append({**base, "variant": "original_headline", "text": str(row["original_text"])})
            jobs.append({**base, "variant": "shuffled_edit", "text": str(row["shuffled_text"])})
    return jobs


def measure_jobs(provider: Any, jobs: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], dict[str, float]]:
    from mesh_signals import compute_signals, split_setup_punchline
    import torch

    output: list[dict[str, Any]] = []
    variant_seconds: dict[str, float] = {}
    start = time.perf_counter()
    for index, job in enumerate(jobs):
        item_start = time.perf_counter()
        local_seed = SEED + index
        random.seed(local_seed)
        np.random.seed(local_seed % (2**32 - 1))
        torch.manual_seed(local_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(local_seed)
        setup, punchline = split_setup_punchline(job["text"])
        try:
            signal = compute_signals(provider, setup, punchline, personas=[PERSONA])
            if not signal.measured or signal.profile is None or not signal.profile.measured:
                raise RuntimeError("Gemma teacher-forced logprobs were not measured")
            persona_measured = bool(signal.personas and signal.personas[0].measured)
            record = {
                **job,
                "setup": setup,
                "punchline": punchline,
                "frame_hint": signal.frame_hint,
                "surprise_mean": signal.surprise_mean,
                "resolution": signal.resolution,
                "efficiency": signal.efficiency,
                "bad_surprise": signal.bad_surprise,
                "S_score": signal.surprise_score,
                "R_score": signal.resolution_score,
                "E_score": signal.efficiency_score,
                "B_score": signal.benign_score,
                "full_score": signal.laugh_score,
                "failure_mode": signal.failure_mode,
                "gemma_logprobs_measured": True,
                "bad_surprise_measured": persona_measured,
                "persona": PERSONA,
                "persona_note": signal.personas[0].note if signal.personas else "",
                "seed": local_seed,
                "seconds": time.perf_counter() - item_start,
                "error": None,
            }
        except Exception as exc:
            record = {
                **job,
                "setup": setup,
                "punchline": punchline,
                "seed": local_seed,
                "seconds": time.perf_counter() - item_start,
                "error": f"{type(exc).__name__}: {exc}",
            }
        output.append(record)
        variant_seconds[job["variant"]] = variant_seconds.get(job["variant"], 0.0) + float(record["seconds"])
        with Path("ablation_rows.jsonl").open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(record, ensure_ascii=False) + "\n")
        if (index + 1) % 20 == 0 or index + 1 == len(jobs):
            print(f"measured {index + 1}/{len(jobs)} jobs in {time.perf_counter() - start:.1f}s", flush=True)
    return output, variant_seconds


def select_failure_cases(frame: pd.DataFrame) -> pd.DataFrame:
    human = frame[frame["variant"] == "human_edit"].copy()
    human["human_rank"] = human["grade"].rank(method="average", pct=True)
    human["model_rank"] = human["full_score"].rank(method="average", pct=True)
    human["rank_error"] = human["model_rank"] - human["human_rank"]
    false_positive = human.nlargest(3, "rank_error").assign(case_type="model_high_human_low")
    false_negative = human.nsmallest(3, "rank_error").assign(case_type="model_low_human_high")
    shuffled = frame[frame["variant"] == "shuffled_edit"].nlargest(2, "full_score").assign(case_type="shuffled_control_false_positive")
    bad_surprise = human.nlargest(2, "bad_surprise").assign(case_type="highest_bad_surprise_risk")
    selected = pd.concat([false_positive, false_negative, shuffled, bad_surprise], ignore_index=True)
    columns = [
        "case_type", "id", "variant", "grade", "text", "surprise_mean", "resolution",
        "efficiency", "bad_surprise", "full_score", "failure_mode", "frame_hint", "persona_note",
    ]
    return selected[columns]


def failure_markdown(failures: pd.DataFrame) -> str:
    columns = list(failures.columns)
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    for _, row in failures.iterrows():
        values = []
        for column in columns:
            value = str(row[column]).replace("\n", " ").replace("|", "\\|")
            values.append(value)
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines) + "\n"


def make_figure(human: pd.DataFrame, ablation: dict[str, Any], controls: dict[str, Any], failures: pd.DataFrame) -> None:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes[0, 0].scatter(human["full_score"], human["grade"], alpha=0.6, s=28, color="#5B5BD6")
    axes[0, 0].set(xlabel="fixed S/R/E/B score", ylabel="human meanGrade", title="Human validation")

    names = ["full_SREB", "without_S", "without_R", "without_E", "without_B", "only_S", "only_R", "only_E", "only_B"]
    rhos = [ablation[name]["spearman"] for name in names]
    colors = ["#1F9D8A" if name == "full_SREB" else "#8FA3BF" for name in names]
    axes[0, 1].barh(names[::-1], rhos[::-1], color=colors[::-1])
    axes[0, 1].axvline(0, color="black", linewidth=0.8)
    axes[0, 1].set(xlabel="Spearman rho", title="Predeclared component ablation")

    variants = ["human_edit", "original_headline", "shuffled_edit"]
    means = [controls["variant_means"][variant]["full_score"] for variant in variants]
    axes[1, 0].bar(["human edit", "original", "shuffled"], means, color=["#1F9D8A", "#C6CBD3", "#E07A5F"])
    axes[1, 0].set(ylabel="mean fixed score", title=f"Paired controls (n={controls['n_complete_sets']})")

    table_rows = []
    for _, row in failures.head(5).iterrows():
        text = str(row["text"]).replace("\n", " ")
        table_rows.append([row["case_type"][:18], f"{row['grade']:.1f}", f"{row['full_score']:.1f}", text[:48]])
    axes[1, 1].axis("off")
    table = axes[1, 1].table(cellText=table_rows, colLabels=["case", "human", "model", "text"], loc="center", cellLoc="left")
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.5)
    axes[1, 1].set_title("Failure cases (not hidden)")

    fig.suptitle("HumorVibes S/R/E/B validation and failure court", fontsize=16)
    fig.tight_layout()
    fig.savefig("ablation_failure_figure.png", dpi=180, bbox_inches="tight")
    plt.close(fig)


def main() -> None:
    started_utc = utc_now()
    wall_start = time.perf_counter()
    Path("ablation_rows.jsonl").unlink(missing_ok=True)
    random.seed(SEED)
    np.random.seed(SEED)

    provider, model_evidence = load_provider()
    print("Gemma evidence:", json.dumps(model_evidence, indent=2), flush=True)
    sampled, data_evidence = load_humicroedit()
    jobs = measurement_jobs(sampled)
    print(f"measurement jobs: {len(jobs)} ({N_HUMAN} human + {2 * N_CONTROL} paired controls)", flush=True)
    rows, variant_seconds = measure_jobs(provider, jobs)
    frame = pd.DataFrame(rows)
    successful = frame[frame["error"].isna()].copy()
    completion_rate = len(successful) / len(frame)
    if completion_rate < 0.95:
        raise RuntimeError(f"Measurement completion rate {completion_rate:.3f} is below 0.95")
    human = successful[successful["variant"] == "human_edit"].copy()
    if len(human) < int(0.95 * N_HUMAN):
        raise RuntimeError("Human-grade measurement coverage is below 95%")

    ablation = component_ablation(human)
    controls = paired_control_court(successful)
    failures = select_failure_cases(successful)
    failures.to_csv("failure_cases.csv", index=False)
    Path("failure_cases.md").write_text(failure_markdown(failures), encoding="utf-8")
    make_figure(human, ablation, controls, failures)

    bad_surprise_coverage = float(human["bad_surprise_measured"].mean())
    summary = {
        "schema": "humorvibes_ablation_v1",
        "status": "complete",
        "external_submission_made": False,
        "sample": {
            "human_rows": int(len(human)),
            "control_sets": int(controls["n_complete_sets"]),
            "measurement_jobs": int(len(frame)),
            "successful_jobs": int(len(successful)),
            "completion_rate": completion_rate,
            "bad_surprise_judge_coverage_on_human": bad_surprise_coverage,
        },
        "metric": {
            "fixed_weights": WEIGHTS,
            "bad_surprise_persona": PERSONA,
            "ablation": ablation,
        },
        "paired_controls": controls,
        "failure_case_count": int(len(failures)),
        "limitations": [
            "Humicroedit headlines are not setup/punchline jokes; edit position often falls outside the inferred punchline span.",
            "Human grades are not persona-specific, while B is measured for one declared broad-news persona.",
            "This is one deterministic sample and one Gemma instrument; confidence intervals quantify item sampling, not model-family uncertainty.",
        ],
    }
    json_dump("ablation_summary.json", summary)

    finished_utc = utc_now()
    runtime = {
        "schema": "humorvibes_runtime_receipt_v1",
        "status": "complete",
        "kernel_id": KERNEL_ID,
        "kernel_private_at_run": True,
        "external_submission_made": False,
        "started_utc": started_utc,
        "finished_utc": finished_utc,
        "wall_seconds": time.perf_counter() - wall_start,
        "variant_seconds": variant_seconds,
        "seconds_per_successful_job": (time.perf_counter() - wall_start) / len(successful),
        "seed": SEED,
        "bootstrap_rounds": BOOTSTRAPS,
        "model": model_evidence,
        "data": data_evidence,
        "environment": {
            "python": sys.version,
            "platform": platform.platform(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
        },
        "outputs": {},
        "reproduction": "Run the private Kaggle notebook humorvibes-ablation-court; it vendors the hashed signal source and attaches the pinned Gemma model source.",
    }
    for output in ("ablation_rows.jsonl", "ablation_summary.json", "failure_cases.csv", "failure_cases.md", "ablation_failure_figure.png"):
        runtime["outputs"][output] = {"bytes": Path(output).stat().st_size, "sha256": sha256_file(output)}
    json_dump("runtime_receipt.json", runtime)

    print("\n=== ABLATION ===")
    for name, result in ablation.items():
        print(f"{name:14s} rho={result['spearman']:+.4f} 95%CI={result['spearman_bootstrap_95ci']}")
    print("\n=== PAIRED CONTROLS ===")
    print(json.dumps(controls, indent=2))
    print("\n=== RUNTIME ===")
    print(json.dumps(runtime, indent=2))


if __name__ == "__main__":
    main()
